# Does onset/offset MAD mechanically depend on OFF size?

`onset_mad` / `offset_mad` are used in Supplementary Figure S2b as estimators of a latent
quantity: how synchronously a population entered or left an OFF period. They are also
strongly correlated with event size. That correlation has two readings, and they call
for opposite treatment:

- Generative (mediator). Larger OFF periods genuinely have more dispersed edges. Size
  then mediates the condition effect, and adjusting for it removes real signal, just as
  a condition effect on `area` is not adjusted for area's correlation with span and
  duration. No adjustment is licensed.
- Mechanical (measurement confound). The estimator's value changes with event size even
  when the latent edge dispersion does not. Size then confounds the measurement, and
  adjustment, or a better estimator, is required.

The correlation alone cannot tell them apart. Part 1 apportions the observed MAD-vs-size
relation between the two accounts, Part 2 works out what that implies for the S2b
contrasts, and Part 3 resolves the secondary explanations.

Everything reads the event-level parquets, which are GitHub Release assets rather than
committed: `release_data.get_event_table_path` prefers a copy in `r-offp/inst/extdata`
and fetches to a cache otherwise. No NFS is needed either way.


In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cnpix_local_sleep.morphological import edge_synchrony_validation as esv

OUT = pathlib.Path("outputs")
OUT.mkdir(exist_ok=True)

# Validated categorical palette (dataviz skill: blue / orange / aqua / violet;
# worst all-pairs CVD dE 9.2, normal-vision 16.3).
BLUE, ORANGE, AQUA, VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"
INK, MUTED = "#1a1a19", "#8a8a80"

plt.rcParams.update({
    "figure.dpi": 120, "savefig.bbox": "tight", "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "grid.color": "#e8e8e4", "grid.linewidth": 0.6, "axes.grid": True,
    "axes.axisbelow": True, "legend.frameon": False,
})

MS = 1e3  # seconds -> ms

## 0. Baseline: reproduce exactly what S2b is built from

Nothing downstream is trustworthy until the recomputed cell means match the shipped
`summarized_full48h_llas_offs.parquet` that r-offp actually models.

In [ ]:
events = esv.load_events("llas")
events = esv.add_floor_free_rvs(events)
events["log_median_duration"] = np.log(events["median_duration"])
print(f"{len(events):,} events, {events['combo'].nunique()} subject-structure combos")

recomputed = (
    events.groupby(["subject", "probe", "structure", "condition"], observed=True)
    .agg(mean_onset_mad=("onset_mad", "mean"),
         mean_offset_mad=("offset_mad", "mean"),
         count=("onset_mad", "size"))
    .reset_index()
)
shipped = pd.read_parquet(esv.R_OFFP_EXTDATA / "summarized_full48h_llas_offs.parquet")
merged = shipped.merge(recomputed, on=["subject", "probe", "structure", "condition"],
                       suffixes=("_shipped", "_recomputed"))
assert len(merged) == len(shipped)
for col in ["mean_onset_mad", "mean_offset_mad", "count"]:
    delta = (merged[f"{col}_shipped"] - merged[f"{col}_recomputed"]).abs().max()
    print(f"  {col:18s} max |shipped - recomputed| = {delta:.3e}")
    assert delta == 0.0
print("BASELINE GATE PASSED")

In [ ]:
nrem = events[events["condition"].isin(esv.NREM_CONDITIONS)].copy()
wake = events[events["condition"].isin(esv.WAKE_CONDITIONS)].copy()

panel = (
    recomputed[recomputed.condition.isin(esv.NREM_CONDITIONS)]
    .pivot_table(index=["subject", "probe", "structure"], columns="condition",
                 values="mean_onset_mad")
    .reindex(columns=esv.NREM_CONDITIONS)
)
print("Published S2b 'All OFFs' onset panel, combo-averaged (ms):")
print((panel.mean() * MS).round(3).to_string())
print(f"n = {len(panel)} subject-structure pairs (manuscript says 29)")

## Part 1: Apportioning the MAD-vs-size relation

### 1.1 The analytic floor

`span` is not merely correlated with the sample size MAD is computed over; it *is*
that sample size. `scipy.ndimage.label` uses four-connectivity, so every channel
between an event's deepest and most superficial channel carries at least one of its
pixels, and `n_channels == span / 20 + 1` exactly.

`MAD = median(|t - median(t)|)` is zero iff enough deviations are exactly zero:
`n // 2 + 1` of them (`mad_zero_min_ties`).

The detector's spatial *opening* uses a `(1, 4)` structuring element, so after
cleaning the ON channels at any single time sample are a union of runs of at least 4
consecutive channels; the following closing is extensive and can only add pixels. So
at a blob's earliest time sample (by definition the earliest ON time of every
channel in it) at least 4 channels turn on together. Whenever 4 ties suffice, MAD is
identically zero no matter how dispersed the other channels are:

`4 >= n // 2 + 1` iff `n <= 7`.

In [ ]:
esv.assert_production_clean_opts()
print("production cleaning:", esv.PRODUCTION_CLEAN_OPTS)

forced = esv.mad_zero_forced_max_n()
print(f"ties needed to force MAD=0, n=6..12: {esv.mad_zero_min_ties(range(6, 13))}")
print(f"=> MAD is forced to zero for every event with n_channels <= {forced}")

floored = nrem[nrem["n_channels"] <= forced]
print(f"\n{len(floored):,} NREM events at n <= {forced}")
print(f"  max onset_mad  = {floored['onset_mad'].max():.6f} s")
print(f"  max offset_mad = {floored['offset_mad'].max():.6f} s")
print(f"  mean onset_jitter (detrended) = {floored['onset_jitter'].mean() * MS:.2f} ms")
print(f"  mean |slope| * span           = {floored['onset_ramp'].mean() * MS:.2f} ms")

Those events carry 2 ms of detrended jitter and a 6 ms propagation ramp, and MAD reports
exactly zero for every one of them. At `n <= 7` the statistic is not a noisy measurement
of asynchrony; it carries no information about it at all.

The floor then releases gradually. Its fingerprint is a parity sawtooth: because the tie
requirement is `n // 2 + 1`, odd `n` needs the same number of ties as the even `n` below
it but has one more channel to supply them, so `P(MAD = 0)` rises from n=8 to n=9, from
n=10 to n=11, and so on. No biological process produces that.


In [ ]:
curve = esv.edge_floor_curve(nrem, "onset")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
ax = axes[0]
ax.plot(curve.n_channels, curve.p_mad_zero, "-o", ms=4, color=BLUE, label="onset MAD")
ax.plot(curve.n_channels, curve.p_jitter_zero, "-o", ms=4, color=ORANGE,
        label="onset jitter (detrended)")
ax.axvspan(curve.n_channels.min() - 0.5, forced + 0.5, color=MUTED, alpha=0.12)
ax.text(forced + 1, 0.93, f"forced zero\n(n $\\leq$ {forced})", fontsize=8, color=INK)
ax.set(xlabel="channels spanned (n = span/20 + 1)", ylabel="P(statistic = 0)",
       xlim=(5, 45))
ax.legend(loc="upper right")

ax = axes[1]
ax.plot(curve.n_channels, curve.mean_mad * MS, "-o", ms=4, color=BLUE, label="MAD")
ax.plot(curve.n_channels, curve.mean_jitter * MS, "-o", ms=4, color=ORANGE,
        label="jitter")
ax.plot(curve.n_channels, curve.mean_ramp * MS, "-o", ms=4, color=AQUA,
        label="|slope| $\\times$ span")
ax.axvspan(curve.n_channels.min() - 0.5, forced + 0.5, color=MUTED, alpha=0.12)
ax.set(xlabel="channels spanned", ylabel="ms", xlim=(5, 45))
ax.legend(loc="upper left")
fig.suptitle("The MAD floor and its release, NREM events", y=1.02, fontsize=10)
fig.savefig(OUT / "fig1_floor.svg")
plt.show()

print(curve[curve.n_channels <= 14][
    ["n_channels", "p_mad_zero", "mean_mad", "n_events"]].round(4).to_string(index=False))

### 1.2 Saturation

Both generative accounts predict MAD keeps growing with size: traversal time at a
fixed wave speed is proportional to span, and "bigger events are more ragged" has no
reason to stop. A floor that releases with `n` predicts a rise followed by a plateau.

In [ ]:
tail = curve[curve.n_channels >= 20]
print("mean onset MAD (ms) over the upper span range:")
print(tail[["n_channels", "mean_mad", "n_events"]]
      .assign(mean_mad=lambda d: d.mean_mad * MS).round(3).to_string(index=False))
lo = curve.set_index("n_channels").loc[26, "mean_mad"] * MS
hi = curve.set_index("n_channels")["mean_mad"].loc[41:].mean() * MS
print(f"\nn=26: {lo:.2f} ms   n>=41 mean: {hi:.2f} ms   "
      f"while span more than doubles over that range")

### 1.3 Forward simulation through the real detector

The quantitative apportionment. Synthetic events are drawn with a known,
size-independent latent edge dispersion and propagation slope, then pushed through the
production code path: `clean_binary_mask` with the shipped options,
`scipy.ndimage.label`, `get_off_properties`. Any size dependence that comes back is
mechanical by construction.

This is a lower bound on the mechanical component. It does not model the 30 ms temporal
median filter applied to the traces upstream of thresholding, which smooths edges
further.


In [ ]:
demo = esv.simulate_detector_edges(
    [6, 8, 11, 16, 21, 26, 41], sigma_ms=6.0, duration_ms=80.0, n_events=200, seed=0)
summary = demo.groupby("n_channels").agg(
    p_mad_zero=("onset_mad", lambda s: (s == 0).mean()),
    mean_mad_ms=("onset_mad", lambda s: s.mean() * MS),
    mean_jitter_ms=("onset_jitter", lambda s: s.mean() * MS),
    n=("onset_mad", "size"))
print("latent dispersion held at 6 ms, no propagation:")
print(summary[summary.n >= 30].round(3).to_string())

With the latent value pinned, the recovered MAD still climbs from 0 to ~2 ms. The floor
at `n <= 7` reproduces independently of the analytic derivation.

Next: can one size-independent (dispersion, propagation-slope-spread) pair reproduce the
whole observed curve at once? If so, nothing about the underlying edge behaviour needs
to change with size to explain what S2b's statistic does.


In [ ]:
GRID_CACHE = OUT / "mechanical_surface_grid.csv"
FIT_CACHE = OUT / "mechanical_surface_fit.csv"
FIT_N = [6, 8, 11, 16, 21, 26, 31, 41]

if GRID_CACHE.exists() and FIT_CACHE.exists():
    grid = pd.read_csv(GRID_CACHE)
    fit = pd.read_csv(FIT_CACHE, index_col=0)
else:
    target = curve[curve.n_channels.isin(FIT_N)]
    grid, best = esv.fit_mechanical_surface(
        target,
        sigma_grid=[10.0, 15.0, 22.0, 30.0, 40.0, 55.0],
        slope_sd_grid=[0.0, 30.0, 50.0, 70.0, 100.0],
        duration_ms=55.0, duration_sd_ms=25.0, n_events=250, seed=5)
    fit = pd.DataFrame({"observed": best["observed"], "predicted": best["predicted"]})
    fit = fit.dropna().loc[fit.index.isin(FIT_N)]
    grid.to_csv(GRID_CACHE, index=False)
    fit.to_csv(FIT_CACHE)

winner = grid.sort_values("rmse").iloc[0]
print(f"best single size-independent pair: sigma = {winner.sigma_ms:.0f} ms, "
      f"slope sd = {winner.slope_sd_us_per_um:.0f} us/um, "
      f"RMSE = {winner.rmse * MS:.2f} ms")
print("\nRMSE (ms) surface:")
print((grid.pivot(index="sigma_ms", columns="slope_sd_us_per_um", values="rmse") * MS)
      .round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(4.4, 3.2))
ax.plot(fit.index, fit["observed"] * MS, "-o", ms=5, color=BLUE, label="observed")
ax.plot(fit.index, fit["predicted"] * MS, "--s", ms=5, color=ORANGE,
        label="one size-independent\nmechanical parameter pair")
ax.axvspan(5, forced + 0.5, color=MUTED, alpha=0.12)
ax.text(forced + 0.8, 8.4, "floored,\nexcluded from fit", fontsize=8, color=INK)
ax.set(xlabel="channels spanned", ylabel="mean onset MAD (ms)")
ax.legend(loc="lower right")
ax.set_title("A constant latent asynchrony reproduces the size curve", fontsize=10)
fig.savefig(OUT / "fig2_mechanical_fit.svg")
plt.show()

Part 1 verdict. A single latent dispersion and a single spread of propagation slopes,
neither depending on event size, reproduce the observed sevenfold rise in mean MAD
across the span range. No generative "bigger events are genuinely more asynchronous"
component is needed. The MAD-vs-size relation is a property of the measurement rather
than of the events, so adjustment is licensed.


### 1.4 The full chain, filters included, and the duration control

`simulate_detector_edges` (1.3) injects a mask directly, so it models only the
morphological cleaning. Production reaches that mask through a 20 Hz Gaussian low-pass
and a 30 ms temporal median filter first. `simulate_trace_level_edges` adds both,
generating a synthetic MUA envelope and thresholding it the way `detect_full` does.

This matters for one specific question. Channel count has an obvious mechanical route to
MAD. Duration does not, but both filters act over a fixed time constant, so in principle
the effect of a filter on a recovered edge could depend on how long the event lasts.
That is the only way duration could earn a place in the adjustment, so it has to be
tested.


In [ ]:
esv.assert_production_trace_opts()

NS_DUR = [11, 16, 26, 41]
DURATIONS = [40.0, 60.0, 90.0, 130.0, 190.0, 280.0]

rows = []
for duration_ms in DURATIONS:
    sim = esv.simulate_trace_level_edges(
        NS_DUR, sigma_ms=6.0, duration_ms=duration_ms, n_events=250, seed=1
    )
    intact = sim[sim["n_channels"] == sim["requested_n_channels"]]
    rows.append(
        intact.groupby("n_channels")["onset_mad"].mean().mul(1e3).rename(duration_ms)
    )
sim_duration = pd.concat(rows, axis=1).reindex(NS_DUR)
sim_duration.columns.name = "duration_ms"
print("simulated mean onset MAD (ms), latent sigma pinned at 6 ms:")
sim_duration.round(3)

Flat. Now the same statistic in the data, at the same fixed channel counts.

In [ ]:
nrem_events = events[events["condition"].isin(esv.NREM_CONDITIONS)]
bins = [0] + [d * 1e-3 for d in DURATIONS[:-1]] + [np.inf]
emp_duration = (
    nrem_events[nrem_events["n_channels"].isin(NS_DUR)]
    .assign(dur_bin=lambda d: pd.cut(d["median_duration"], bins))
    .groupby(["n_channels", "dur_bin"], observed=True)["onset_mad"]
    .agg(["mean", "size"])
)
emp_duration["mean"] *= 1e3
print("observed mean onset MAD (ms) by duration bin, WITHIN fixed channel count:")
emp_duration[emp_duration["size"] >= 100].round(2)

A threefold monotone rise that the estimator does not produce. By the same argument used
for span in 1.1-1.3, that relation is generative: longer OFF periods really do have more
dispersed edges. Duration therefore mediates the condition effect rather than confounding
the measurement, and it must not enter the adjustment.

Two ablations close the obvious loopholes: the median filter is genuinely running, and
it is nonetheless edge-preserving at realistic event lengths, which is why it cannot
carry a duration effect.


In [ ]:
no_clean = {"n_samples_connect": None, "n_samples_clean": None,
            "n_channels_clean": None, "n_channels_connect": None}
shared = dict(sigma_ms=2.0, n_events=20, seed=9, clean_opts=no_clean)

for duration_ms in (20.0, 120.0):
    survived = {}
    for apply_median in (True, False):
        out = esv.simulate_trace_level_edges(
            [16], duration_ms=duration_ms, apply_median=apply_median, **shared
        )
        big = out[out["span"] >= 200]
        survived[apply_median] = (len(big), big["onset_mad"].mean() * 1e3)
    print(f"duration {duration_ms:5.0f} ms   median ON: {survived[True]}   "
          f"median OFF: {survived[False]}")

print("\nA 20 ms dip is shorter than the 30 ms window and does not survive it, so the")
print("filter is applied. At 120 ms the two are identical, so it is edge-preserving.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4), sharey=True)
# One hue, light -> dark: channel count is a magnitude, not an identity.
SHADES = dict(zip(NS_DUR, ["#bcd6f4", "#7aaeea", "#3a7fd0", "#12508f"]))
for n in NS_DUR:
    axes[0].plot(sim_duration.columns, sim_duration.loc[n], "-o", color=SHADES[n], label=f"{n} ch")
    sub = emp_duration.loc[n]
    sub = sub[sub["size"] >= 100]
    axes[1].plot([b.mid * 1e3 for b in sub.index], sub["mean"], "-o", color=SHADES[n], label=f"{n} ch")
axes[0].set_title("Simulated: latent dispersion held fixed")
axes[1].set_title("Observed: same events in the data")
for ax in axes:
    ax.set_xlabel("OFF median duration (ms)")
    ax.set_xlim(20, 300)
axes[0].set_ylabel("mean onset MAD (ms)")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "fig5_duration_nb.svg")
sim_duration.to_csv(OUT / "duration_invariance_simulated.csv")

## Part 2: What that implies for the S2b contrasts

Three estimators of the condition effect at fixed size, reported head to head.

In [ ]:
STRATA = ["nbin", "dbin"]
for frame in (events, nrem, wake):
    frame["nbin"] = pd.cut(frame["n_channels"],
                           [5, 7, 9, 11, 14, 18, 24, 32, 44, 10_000], labels=False)
    frame["dbin"] = pd.qcut(frame["median_duration"], 5, labels=False,
                            duplicates="drop")

rows = []
for state, frame, conditions in [("NREM", nrem, esv.NREM_CONDITIONS),
                                 ("Wake", wake, esv.WAKE_CONDITIONS)]:
    for label, subset in [("All OFFs", frame),
                          ("Small", frame[frame.size_class == "Small"]),
                          ("Medium+Large", frame[frame.size_class == "Medium+Large"])]:
        for edge in ("onset", "offset"):
            std = esv.standardize_cell_means(subset, f"{edge}_mad", STRATA)
            for kind in ("raw", "standardized"):
                means = (std.pivot(index="combo", columns="condition", values=kind)
                         .reindex(columns=conditions).mean() * MS)
                rows.append({"state": state, "class": label, "edge": edge,
                             "kind": kind, "dropped": std.frac_dropped.mean(),
                             **means.to_dict()})
standardized = pd.DataFrame(rows)
print("Estimator 1 - direct standardization to a common span x duration mix")
print(standardized[standardized.state == "NREM"].drop(columns=["state"] +
      esv.WAKE_CONDITIONS, errors="ignore").round(3).to_string(index=False))

In [ ]:
print("Wake (note the dropped-event fraction: wake events occupy a much narrower")
print("size range, so common support is a real restriction here)")
print(standardized[standardized.state == "Wake"]
      .drop(columns=["state"] + esv.NREM_CONDITIONS, errors="ignore")
      .round(3).to_string(index=False))

In [ ]:
# Estimator 2 - event-level covariate model, pooled across combos by DerSimonian-Laird.
def contrasts(subset, value, reference, scale=MS):
    out = []
    for tag, adjusted in [("unadjusted", False), ("adjusted", True)]:
        # Channel count only. Duration is a mediator (1.4), so adjusting for it
        # would remove real signal; bout-clustered SEs handle the temporal
        # dependence measured in 3.5.
        _, pooled = esv.fit_event_level(
            subset, value, reference=reference,
            n_channel_factor=adjusted, covariates=None,
            cluster_seconds=60.0, min_events=150)
        if pooled.empty:
            continue
        for _, row in pooled.iterrows():
            out.append({"model": tag, "contrast": f"{row.contrast} - {reference}",
                        "estimate": row.estimate * scale,
                        "ci_lo": row.ci_lo * scale, "ci_hi": row.ci_hi * scale,
                        "p": row.p, "k": int(row.k)})
    return pd.DataFrame(out)


REF_NREM, REF_WAKE = "Early.REC.NREM.Match", "Early.NOD.Wake"
event_level = []
for label, subset in [("All OFFs", nrem),
                      ("Small", nrem[nrem.size_class == "Small"]),
                      ("Medium+Large", nrem[nrem.size_class == "Medium+Large"])]:
    for edge in ("onset", "offset"):
        block = contrasts(subset, f"{edge}_mad", REF_NREM)
        event_level.append(block.assign(**{"class": label, "edge": edge}))
event_level = pd.concat(event_level, ignore_index=True)

view = event_level[event_level.contrast.str.startswith("Early.REC.NREM ")]
print("Estimator 2 - NREM.Rebound (Early.REC.NREM vs circadian Match), ms")
print("negative = MORE synchronous under high sleep pressure")
print(view[["class", "edge", "model", "estimate", "ci_lo", "ci_hi", "p"]]
      .round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.0), sharey=True)
for ax, edge in zip(axes, ("onset", "offset")):
    block = view[view.edge == edge]
    classes = ["All OFFs", "Small", "Medium+Large"]
    for offset, (tag, color) in enumerate([("unadjusted", ORANGE), ("adjusted", BLUE)]):
        sub = block[block.model == tag].set_index("class").reindex(classes)
        y = np.arange(len(classes)) + (offset - 0.5) * 0.22
        ax.errorbar(sub.estimate, y,
                    xerr=[sub.estimate - sub.ci_lo, sub.ci_hi - sub.estimate],
                    fmt="o", ms=5, lw=1.6, color=color, label=tag)
    ax.axvline(0, color=MUTED, lw=1)
    ax.set_yticks(np.arange(len(classes)), classes)
    ax.set(xlabel="Early.REC.NREM - Match (ms)", title=f"{edge} edge")
    ax.grid(axis="y", visible=False)
axes[0].invert_yaxis()  # shared y: flips both, read top-to-bottom
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.20))
fig.suptitle("Adjusting for event size flips or erases every published MAD contrast",
             y=1.04, fontsize=10)
fig.savefig(OUT / "fig3_contrasts.svg")
plt.show()

In [ ]:
# Estimator 3 - the floor-free and scale-free alternatives, all already exported.
SPECS = [("onset_mad", MS, "ms"), ("onset_jitter", MS, "ms"),
         ("onset_abs_slope", 1e6, "us/um"), ("onset_r2", 1.0, "-"),
         ("onset_mad_per_um", 1e6, "us/um"), ("onset_mad_rel_duration", 1.0, "-")]
large = nrem[nrem.size_class == "Medium+Large"]
rows = []
for value, scale, unit in SPECS:
    block = contrasts(large, value, REF_NREM, scale=scale)
    block = block[block.contrast.str.startswith("Early.REC.NREM ")]
    for _, row in block.iterrows():
        rows.append({"statistic": value, "unit": unit, "model": row.model,
                     "estimate": row.estimate, "p": row.p})
alternatives = pd.DataFrame(rows).pivot_table(
    index=["statistic", "unit"], columns="model", values=["estimate", "p"])
print("Estimator 3 - NREM.Rebound in Medium+Large events, by statistic")
print("negative = more synchronous / faster propagation under high sleep pressure")
print(alternatives.round(4).to_string())

Every scale-free version of the same measurement moves in the same direction once size
is controlled.

None of them is a validated drop-in replacement, though. The same forward simulation
that convicted MAD convicts them too: at a constant latent dispersion, `jitter` rises
with span and `|slope|` falls, since a regression slope fitted to few noisy points is
unstable. On the real data `jitter` and `|slope|` are additionally floored at small `n`,
and the MAD ratios inherit MAD's floor exactly.


In [ ]:
fixed = esv.simulate_detector_edges(
    [6, 11, 26, 41], sigma_ms=6.0, duration_ms=80.0, n_events=150, seed=0)
print("simulated at a latent dispersion pinned to 6 ms, no propagation:")
print(fixed.groupby("n_channels").agg(
    mean_mad_ms=("onset_mad", lambda s: s.mean() * MS),
    mean_jitter_ms=("onset_jitter", lambda s: s.mean() * MS),
    median_abs_slope_us_um=("onset_slope", lambda s: np.median(np.abs(s)) * 1e6),
).round(3).to_string())

floored_events = nrem[nrem["n_channels"] <= forced]
print("\non the real data, at n <= 7:")
print(f"  P(jitter == 0)           = {(floored_events['onset_jitter'] == 0).mean():.3f}")
print(f"  P(|slope| == 0)          = {(floored_events['onset_slope'] == 0).mean():.3f}")
print(f"  P(r2 undefined)          = {floored_events['onset_r2'].isna().mean():.3f}")
print(f"  max mad/span             = {(floored_events['onset_mad_per_um']).max():.3e}")
print(f"  max mad/median_duration  = {(floored_events['onset_mad_rel_duration']).max():.3e}")

#### Does normalizing by span help?

No. Dividing by a positive number cannot change whether a value is zero, so the hard
floor and the parity sawtooth pass straight through. And because the real dependence is
floor-then-saturate rather than proportional to span, the ratio under-corrects at small
`n` and over-corrects at large `n`, leaving a hump: non-monotone, so a shift in the
size distribution can move it either way.

In [ ]:
ratio = nrem.groupby("n_channels").agg(
    p_mad_zero=("onset_mad", lambda s: (s == 0).mean()),
    p_ratio_zero=("onset_mad_per_um", lambda s: (s == 0).mean()),
    mean_mad_ms=("onset_mad", lambda s: s.mean() * MS),
    mean_ratio_us_um=("onset_mad_per_um", lambda s: s.mean() * 1e6),
    n=("onset_mad", "size"))
ratio = ratio[ratio.n >= 200]
print(ratio.loc[[6, 8, 10, 13, 16, 21, 26, 31, 41]].round(4).to_string())
print(f"\nP(mad/span == 0) identical to P(mad == 0) at every n: "
      f"{np.allclose(ratio.p_mad_zero, ratio.p_ratio_zero)}")

In [ ]:
# Wake, where the largest raw effect in S2b lives.
wake_rows = []
for value, scale, unit in [("onset_mad", MS, "ms"), ("offset_mad", MS, "ms"),
                           ("onset_jitter", MS, "ms"),
                           ("onset_abs_slope", 1e6, "us/um")]:
    block = contrasts(wake, value, REF_WAKE, scale=scale)
    for _, row in block.iterrows():
        wake_rows.append({"statistic": value, "unit": unit, "model": row.model,
                          "estimate": row.estimate, "p": row.p, "k": row.k})
print("NOD.Incline (Late.NOD.Wake vs Early.NOD.Wake)")
print(pd.DataFrame(wake_rows).round(4).to_string(index=False))

## Part 3: Secondary explanations

### 3.1 Composition across the size partition

"All OFFs" averages two populations whose mean MAD differs about fivefold, with mixing
weights that move with condition. This is a decomposition of the published number, not a
correction: holding class weights fixed removes the gap, holding class means fixed
reproduces it.


In [ ]:
mixture = esv.decompose_mixture(nrem, "onset_mad")
summary = (mixture.groupby("condition", observed=True)[
    ["observed", "fixed_weights", "fixed_class_means", "frac_Medium+Large"]]
    .mean().reindex(esv.NREM_CONDITIONS))
summary[["observed", "fixed_weights", "fixed_class_means"]] *= MS
print("Mean onset MAD (ms) and the Medium+Large share, per condition")
print(summary.round(3).to_string())
print("\nclass means (ms): " + ", ".join(
    f"{k}={v * MS:.2f}" for k, v in
    nrem.groupby("size_class", observed=True)["onset_mad"].mean().items()))

### 3.2 Zero-inflation: where does the published mean actually move?

If the contrast is driven by the floor, it should live in the fraction of floored events
rather than in the events that escape the floor.


In [ ]:
nrem["mad_is_zero"] = (nrem["onset_mad"] == 0).astype(float)
zero_share = contrasts(nrem, "mad_is_zero", REF_NREM, scale=1.0)
nonzero = contrasts(nrem[nrem.onset_mad > 0], "onset_mad", REF_NREM)
print("P(onset_mad == 0):")
print(zero_share[zero_share.contrast.str.startswith("Early.REC.NREM ")]
      .round(4).to_string(index=False))
print("\nE[onset_mad | onset_mad > 0], ms:")
print(nonzero[nonzero.contrast.str.startswith("Early.REC.NREM ")]
      .round(4).to_string(index=False))

### 3.3 Partition as a collider

`Small` and `Medium+Large` are defined by thresholds on span, median duration and
total duration, exactly the quantities that drive MAD. Conditioning on class is
conditioning on a collider, so the published contrast should depend on where the
arbitrary cut is placed. The adjusted contrast should not.

In [ ]:
rows = []
for cut in [150, 200, 250, 300, 400]:
    block = contrasts(nrem[nrem.span >= cut], "onset_mad", REF_NREM)
    block = block[block.contrast.str.startswith("Early.REC.NREM ")]
    for _, row in block.iterrows():
        rows.append({"span_cut_um": cut, "n_events": int((nrem.span >= cut).sum()),
                     "model": row.model, "estimate": row.estimate, "p": row.p})
slide = pd.DataFrame(rows)
print(slide.pivot_table(index=["span_cut_um", "n_events"], columns="model",
                        values=["estimate", "p"]).round(4).to_string())

fig, ax = plt.subplots(figsize=(4.2, 3.0))
for tag, color in [("unadjusted", ORANGE), ("adjusted", BLUE)]:
    sub = slide[slide.model == tag]
    ax.plot(sub.span_cut_um, sub.estimate, "-o", ms=5, color=color, label=tag)
ax.axhline(0, color=MUTED, lw=1)
ax.set(xlabel="stringent span cut (um)", ylabel="Early.REC.NREM - Match (ms)")
ax.legend()
ax.set_title("The published contrast changes sign with an arbitrary cut", fontsize=9)
fig.savefig(OUT / "fig4_collider.svg")
plt.show()

### 3.4 Detection depth, blob holes, and cell weighting

In [ ]:
nrem["tbin"] = pd.qcut(nrem["min_trace"], 4, labels=False, duplicates="drop")
print("mean onset MAD (ms) by min_trace quartile (0 = most completely silenced):")
print((nrem.groupby(["tbin", "size_class"], observed=True)["onset_mad"].mean()
       .unstack() * MS).round(3).to_string())

for strata in (["nbin", "dbin"], ["nbin", "dbin", "tbin"]):
    std = esv.standardize_cell_means(nrem, "onset_mad", strata)
    means = (std.pivot(index="combo", columns="condition", values="standardized")
             .reindex(columns=esv.NREM_CONDITIONS).mean() * MS)
    print(f"\nstandardized over {strata} (dropped {std.frac_dropped.mean():.3f}):")
    print(means.round(3).to_string())

In [ ]:
# Blob holes: per-channel onset/offset are min/max over blob pixels, so temporal
# gaps within a channel inflate the measured edge. Measure the gap directly.
envelope = nrem["median_end_time"] - nrem["median_start_time"]
hole_fraction = 1 - nrem["median_duration"] / envelope
print("hole fraction of the median channel, by condition and class:")
print(nrem.assign(hole=hole_fraction)
      .groupby(["condition", "size_class"], observed=True)["hole"].mean()
      .unstack().round(4).to_string())

### 3.5 Temporal dependence between events, and what an event-level model would cost

The published pipeline aggregates events to one number per cell before modelling, which
neutralizes within-cell temporal dependence by construction. The event-level estimator in
Part 2 does not, and uses OLS standard errors that assume independence. Measure the
dependence directly to see how much that costs.

In [ ]:
records = []
for _, cell in nrem.groupby(["combo", "condition"], observed=True):
    cell = cell.sort_values("start_time")
    values = cell["onset_mad"].to_numpy()
    if len(values) < 500:
        continue
    block = (cell["start_time"] // 60).to_numpy()  # 60-s blocks proxy bout structure
    frame = pd.DataFrame({"x": values, "b": block})
    grand, n_blocks, n = frame.x.mean(), frame.b.nunique(), len(frame)
    if n_blocks < 2 or n <= n_blocks:
        continue
    between = sum(len(s) * (s.x.mean() - grand) ** 2
                  for _, s in frame.groupby("b")) / (n_blocks - 1)
    within = sum(((s.x - s.x.mean()) ** 2).sum()
                 for _, s in frame.groupby("b")) / (n - n_blocks)
    per_block = n / n_blocks
    icc = max(0.0, (between - within) / (between + (per_block - 1) * within))
    records.append({"lag1": np.corrcoef(values[:-1], values[1:])[0, 1],
                    "icc": icc, "per_block": per_block,
                    "deff": 1 + (per_block - 1) * icc})
dependence = pd.DataFrame(records)
inflation = np.sqrt(dependence.deff.median())
print(f"{len(dependence)} cells with >= 500 events")
print(f"  median lag-1 autocorrelation of onset_mad = {dependence.lag1.median():.3f}"
      f"  (IQR {dependence.lag1.quantile(.25):.3f}-{dependence.lag1.quantile(.75):.3f})")
print(f"  median ICC within 60-s blocks             = {dependence.icc.median():.3f}")
print(f"  median events per block                   = {dependence.per_block.median():.0f}")
print(f"  median design effect                      = {dependence.deff.median():.2f}")
print(f"  => OLS standard errors too small by ~{inflation:.2f}x; "
      f"effective n ~{100 / dependence.deff.median():.0f}% of nominal")

import scipy.stats
headline = view[view.model == "adjusted"].copy()
headline["se"] = (headline.ci_hi - headline.ci_lo) / 2 / 1.96
headline["se_infl"] = headline.se * inflation
headline["ci_lo_infl"] = headline.estimate - 1.96 * headline.se_infl
headline["ci_hi_infl"] = headline.estimate + 1.96 * headline.se_infl
headline["p_infl"] = 2 * scipy.stats.norm.sf(
    (headline.estimate / headline.se_infl).abs())
print("\nAdjusted NREM.Rebound with design-effect-widened intervals:")
print(headline[["class", "edge", "estimate", "ci_lo", "ci_hi", "p",
                "ci_lo_infl", "ci_hi_infl", "p_infl"]].round(4).to_string(index=False))

### 3.6 Common support: why standardization works for NREM and not for wake

In [ ]:
support = []
for label, bins, n_dur in [("9 span x 5 dur", [5, 7, 9, 11, 14, 18, 24, 32, 44, 10_000], 5),
                           ("4 span x 3 dur", [5, 9, 14, 24, 10_000], 3),
                           ("2 span x 2 dur", [5, 9, 10_000], 2)]:
    for state, frame in [("NREM", nrem), ("Wake", wake)]:
        block = frame.copy()
        block["sbin"] = pd.cut(block["n_channels"], bins, labels=False)
        block["dbin"] = pd.qcut(block["median_duration"], n_dur, labels=False,
                                duplicates="drop")
        std = esv.standardize_cell_means(block, "onset_mad", ["sbin", "dbin"])
        support.append({"strata": label, "state": state,
                        "dropped_mean": std.frac_dropped.mean(),
                        "dropped_worst_cell": std.frac_dropped.max()})
print(pd.DataFrame(support).pivot(index="strata", columns="state",
                                  values=["dropped_mean", "dropped_worst_cell"])
      .round(3).to_string())

## Part 5: The shipped size-adjusted analysis

Parts 1-3 establish that the MAD-vs-size relation is mechanical, so adjusting for it
is licensed. This part is the estimator that ships: one size-adjustment curve,
estimated once across every cell, applied with a free per-subject-structure
amplitude, and handed to r-offp's existing `crossed_interaction` model as
`adj_mean_{onset,offset}_mad`.

Everything below runs on the Medium+Large (`clas`) class, the only one reported;
section 5.8 of the report says why.


In [ ]:
from scipy import stats as sps
import patsy

ml = events[events["size_class"] == "Medium+Large"].copy()
ml["state"] = np.where(ml["condition"].isin(esv.NREM_CONDITIONS), "nrem", "wake")
ml["unit"] = ml["combo"] + "@" + ml["state"]
ml["cell"] = ml["unit"] + "@" + ml["condition"].astype(str)
nrem_ml = ml[ml["state"] == "nrem"]
wake_ml = ml[ml["state"] == "wake"]
print(f"{len(ml):,} Medium+Large events")
print(f"  NREM {len(nrem_ml):>9,} events in {nrem_ml['combo'].nunique()} combos")
print(f"  wake {len(wake_ml):>9,} events in {wake_ml['combo'].nunique()} combos")


def paired(frame, column, target, reference):
    """Per-combo difference between two conditions, dropping incomplete pairs."""
    wide = frame.pivot_table(
        index=["subject", "probe", "structure"], columns="condition", values=column
    )
    return (wide[target] - wide[reference]).dropna()


CONTRASTS = {
    "NREM.Rebound": ("Early.REC.NREM", "Early.REC.NREM.Match"),
    "NREM.Surge": ("Early.REC.NREM", "Early.BSL.NREM"),
    "NREM.REC.Decline": ("Early.REC.NREM", "Late.REC.NREM"),
    "NOD.Incline": ("Late.NOD.Wake", "Early.NOD.Wake"),
}


### 5.2 The one shared size-adjustment curve

`fit_shared_size_curve` fits `mad = cell mean + amplitude(unit) * f(n)` by alternating
least squares, where a cell is one (combo, state, condition) block and a unit is one
(combo, state) block. Profiling out the cell means is what keeps condition out of the
curve: the contrast being adjusted lives between cells of a unit, and the curve is
estimated only from variation within them.

The shape is shared and only the amplitude is allowed to differ. Two things to check:
that the units really do share a shape, and that it does not matter whether the curve is
estimated from everything or from NREM alone.


In [ ]:
curves = {
    edge: esv.fit_shared_size_curve(ml, f"{edge}_mad", "unit", "cell")
    for edge in ("onset", "offset")
}
nrem_only = esv.fit_shared_size_curve(nrem_ml, "onset_mad", "unit", "cell")

for edge, curve in curves.items():
    amplitude = curve.unit_lambda * MS
    by_state = {s: [u for u in amplitude.index if u.endswith(s)] for s in ("nrem", "wake")}
    print(f"{edge}: {len(curve.curve)} channel counts | "
          f"converged={curve.converged} after {curve.n_iterations} iterations")
    for state, units in by_state.items():
        values = amplitude[units]
        print(f"   amplitude, {state}: median {values.median():.2f} ms "
              f"(range {values.min():.2f}-{values.max():.2f}, "
              f"relative SD {values.std() / values.mean():.2f})")
    correlation = curve.unit_shape_correlation.dropna()
    well_sampled = correlation[curve.unit_events.reindex(correlation.index) >= 60]
    print(f"   per-unit shape correlation with the shared curve: "
          f"median {correlation.median():.3f} over all {len(correlation)} units, "
          f"{well_sampled.median():.3f} over the {len(well_sampled)} with >=60 events "
          f"(min {well_sampled.min():.3f})")

grid = np.arange(11, 80)
agreement = np.corrcoef(curves["onset"].evaluate(grid), nrem_only.evaluate(grid))[0, 1]
print(f"\ncorr(curve from all events, curve from NREM alone) = {agreement:.5f}")

pd.concat(
    [curves["onset"].curve.rename("f_onset"), curves["offset"].curve.rename("f_offset")],
    axis=1,
).rename_axis("n_channels").to_csv(OUT / "shared_size_curve.csv")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, edge in zip(axes, ("onset", "offset"), strict=True):
    curve = curves[edge]
    residual = ml[f"{edge}_mad"] - ml.groupby("cell", observed=True)[f"{edge}_mad"].transform("mean")
    profile = (
        ml.assign(residual=residual)
        .groupby(["unit", "n_channels"], observed=True)
        .agg(residual=("residual", "mean"), n_events=("residual", "size"))
        .reset_index()
    )
    profile = profile[profile["n_events"] >= 50]
    for unit, block in profile.groupby("unit", observed=True):
        amplitude = curve.unit_lambda.get(unit, np.nan)
        if not np.isfinite(amplitude) or amplitude <= 0:
            continue
        ax.plot(block["n_channels"], block["residual"] / amplitude,
                color=MUTED, lw=0.6, alpha=0.35)
    ax.plot(curve.curve.index, curve.curve.to_numpy(), color=BLUE, lw=2.2,
            label="shared curve", zorder=5)
    ax.set_xlim(11, 80)
    ax.set_xlabel("channels spanned")
    ax.set_title(f"{edge} MAD", loc="left")
axes[0].set_ylabel("size effect, amplitude-normalized")
axes[0].set_ylim(-2.5, 3.0)
axes[0].legend(loc="lower right")
fig.tight_layout()
fig.savefig(OUT / "fig6_shared_curve.svg")


### 5.4 The shipped cell means

`compute_adjusted_cell_means` fits the curve once per edge across both states, then
g-computes one adjusted mean per cell: predict every condition over all of that unit's
events, so every condition is scored on one and the same size distribution.

Units too small to identify their own amplitude are given the state's pooled one rather
than being dropped, so the adjusted panel covers exactly the cells the published panel
does. The paired means below equal r-offp's fitted contrasts for the balanced NREM
design; the wake panel is unbalanced, so its authoritative numbers come from the mixed
model in `_output/full48h/clas/wake/`.


In [ ]:
adjusted = esv.compute_adjusted_cell_means(ml)
state = np.where(adjusted["condition"].isin(esv.WAKE_CONDITIONS), "wake", "nrem")
print(adjusted.groupby([state, "onset_size_coding"]).size().to_string(), "\n")

rows = []
for name, (target, reference) in CONTRASTS.items():
    for edge in ("onset", "offset"):
        difference = paired(adjusted, f"{esv.ADJUSTED_PREFIX}mean_{edge}_mad", target, reference)
        unadjusted = paired(adjusted, f"mean_{edge}_mad", target, reference) * MS
        rows.append({
            "contrast": name, "edge": edge, "k_pairs": len(difference),
            "unadjusted_ms": unadjusted.mean(), "adjusted_ms": difference.mean(),
            "paired_t_p": sps.ttest_1samp(difference, 0).pvalue,
        })
shipped = pd.DataFrame(rows)
print(shipped.round(3).to_string(index=False))
adjusted.to_csv(OUT / "adjusted_cell_means.csv", index=False)


### 5.5 Agreement across estimators

Three estimators of the same Medium+Large NREM rebound, sharing almost no assumptions:
marginal standardization (what ships), the event-level two-stage model with
bout-clustered standard errors, and stratified direct standardization on channel count.
Disagreement here would be a warning.


In [ ]:
records = []
for edge in ("onset", "offset"):
    value = f"{edge}_mad"
    marginal = paired(
        adjusted, f"{esv.ADJUSTED_PREFIX}mean_{edge}_mad",
        "Early.REC.NREM", "Early.REC.NREM.Match",
    ).mean()
    _, pooled = esv.fit_event_level(
        nrem_ml, value, reference="Early.REC.NREM.Match", cluster_seconds=60.0
    )
    two_stage = pooled.set_index("contrast").loc["Early.REC.NREM"]
    _, unadjusted_pooled = esv.fit_event_level(
        nrem_ml, value, reference="Early.REC.NREM.Match", n_channel_factor=False
    )
    stratified = esv.standardize_cell_means(nrem_ml, value, ["n_channels"])
    wide = stratified.pivot_table(index="combo", columns="condition", values="standardized")
    records.append({
        "edge": edge,
        "marginal_standardization_ms": marginal,
        "event_level_clustered_ms": two_stage["estimate"] * MS,
        "event_level_p": two_stage["p"],
        "stratified_ms": (wide["Early.REC.NREM"] - wide["Early.REC.NREM.Match"]).mean() * MS,
        "stratified_worst_dropped": stratified["frac_dropped"].max(),
        "event_level_unadjusted_ms": (
            unadjusted_pooled.set_index("contrast").loc["Early.REC.NREM", "estimate"] * MS
        ),
    })
estimator_agreement = pd.DataFrame(records)
print(estimator_agreement.round(3).to_string(index=False))
estimator_agreement.to_csv(OUT / "estimator_agreement.csv", index=False)


### 5.10 Robustness to how size is coded

Every coding is run through the identical estimator, so the rows differ only by the size
term. `size_term="none"` fits no size term at all, which is both the unadjusted contrast
and the hook for an arbitrary coding passed through `covariates`.


In [ ]:
def with_size_bases(frame, spline_df=5):
    """Attach polynomial and natural-spline codings of channel count."""
    work = frame.copy()
    counts = work["n_channels"].to_numpy(dtype=float)
    z = (counts - counts.mean()) / counts.std()
    for power in (1, 2, 3):
        work[f"poly{power}"] = z**power
    basis = patsy.dmatrix(
        f"cr(n, df={spline_df}) - 1", {"n": counts}, return_type="dataframe"
    )
    spline_columns = [f"spline{i}" for i in range(basis.shape[1])]
    work[spline_columns] = basis.to_numpy()
    return work, tuple(spline_columns)


def coding_specs(spline_columns, quantiles=(3, 6, 10)):
    specs = [
        ("unadjusted", dict(size_term="none")),
        ("shared curve, free amplitude (shipped)", {}),
        ("shared curve, pooled amplitude", dict(free_lambda_min_events=10**9)),
        ("per-combo exact factor", dict(size_term="per_combo_factor",
                                        events_per_size_bin=1, min_events=1)),
    ]
    specs += [
        (f"per-combo quantile[{k}]",
         dict(size_term="per_combo_factor", exact_size_levels=False,
              n_size_bins=k, events_per_size_bin=1, min_events=1))
        for k in quantiles
    ]
    specs += [
        ("linear in n", dict(size_term="none", covariates=("poly1",))),
        ("quadratic", dict(size_term="none", covariates=("poly1", "poly2"))),
        ("cubic", dict(size_term="none", covariates=("poly1", "poly2", "poly3"))),
        ("natural spline", dict(size_term="none", covariates=spline_columns)),
    ]
    return specs


def coding_table(frame, contrasts, curves=curves):
    """Every coding through the identical estimator, on one fixed population.

    The shared-curve rows are handed the *production* curve (the one fitted across
    both states in 5.2) rather than refitting it on whatever subset this table is
    built from, so those rows are the shipped estimator restricted to this
    population and nothing else.
    """
    work, spline_columns = with_size_bases(frame)
    rows = []
    for label, kwargs in coding_specs(spline_columns):
        for edge in ("onset", "offset"):
            curve = curves[edge]
            out = esv.standardize_by_regression(
                work, f"{edge}_mad", shared_curve=curve,
                pooled_lambda=esv.pooled_amplitude(work, f"{edge}_mad", curve),
                **kwargs,
            )
            wide = out.pivot_table(index="combo", columns="condition", values="standardized")
            row = {"coding": label, "edge": edge, "k": len(wide)}
            for name, (target, reference) in contrasts.items():
                row[name] = (wide[target] - wide[reference]).mean() * MS
            rows.append(row)
    return pd.DataFrame(rows)


nrem_contrasts = {k: v for k, v in CONTRASTS.items() if k != "NOD.Incline"}
size_coding_robustness = coding_table(nrem_ml, nrem_contrasts)
for edge in ("onset", "offset"):
    print(f"--- NREM {edge} MAD (ms)")
    print(size_coding_robustness[size_coding_robustness["edge"] == edge]
          .drop(columns="edge").round(3).to_string(index=False))
size_coding_robustness.to_csv(OUT / "size_coding_robustness.csv", index=False)


### 5.12 Wake, and the gate that used to shrink it

The wake panel is where the per-combo factor breaks down: a wake unit holds a median
of ~30 Medium+Large events across ~15 distinct channel counts, so a parameter per
count is hopeless and even quantile bins are thin. The earlier estimator handled that
by dropping any unit with fewer than 60 events, which silently removed 25 of the 42
published wake cells, so "adjusted" and "unadjusted" were not being compared on the
same sample.

The table below holds the population fixed at the 16 combos contributing both wake
conditions, so the rows differ only by the size term. The sweep after it shows what
the dropped-unit gate was doing.


In [ ]:
complete = [
    combo for combo, block in wake_ml.groupby("combo", observed=True)
    if block["condition"].nunique() == 2
]
wake_common = wake_ml[wake_ml["combo"].isin(complete)]
print(f"{len(complete)} wake combos contribute both conditions "
      f"({len(wake_common):,} events; median {wake_common.groupby('combo').size().median():.0f} per combo)")

wake_coding = coding_table(wake_common, {"NOD.Incline": CONTRASTS["NOD.Incline"]})
for edge in ("onset", "offset"):
    print(f"\n--- wake {edge} MAD (ms), k = {len(complete)} combos throughout")
    print(wake_coding[wake_coding["edge"] == edge]
          .drop(columns="edge").round(3).to_string(index=False))
wake_coding.to_csv(OUT / "wake_size_coding.csv", index=False)


In [ ]:
gate = []
for threshold in (1, 20, 30, 60, 120):
    for edge in ("onset", "offset"):
        curve = curves[edge]
        out = esv.standardize_by_regression(
            wake_ml, f"{edge}_mad", min_events=threshold, shared_curve=curve,
            pooled_lambda=esv.pooled_amplitude(wake_ml, f"{edge}_mad", curve),
        )
        wide = out.pivot_table(index="combo", columns="condition", values="standardized")
        complete_pairs = wide.dropna()
        gate.append({
            "min_events": threshold, "edge": edge,
            "cells": len(out), "complete_pairs": len(complete_pairs),
            "adjusted_ms": (complete_pairs["Late.NOD.Wake"]
                            - complete_pairs["Early.NOD.Wake"]).mean() * MS,
            "unadjusted_ms": (
                out.pivot_table(index="combo", columns="condition", values="raw")
                .dropna()
                .pipe(lambda w: w["Late.NOD.Wake"] - w["Early.NOD.Wake"])
                .mean() * MS
            ),
        })
gate = pd.DataFrame(gate)
print(gate.round(3).to_string(index=False))
print("\nThe shipped estimator uses min_events=1: nothing is dropped.")
gate.to_csv(OUT / "wake_dropped_unit_gate.csv", index=False)


## Summary

Cache the tables this notebook produced, so the figure script can draw its mechanism
panels without re-running the simulations.

In [ ]:
standardized.to_csv(OUT / "standardized_cell_means.csv", index=False)
event_level.to_csv(OUT / "event_level_contrasts.csv", index=False)
alternatives.to_csv(OUT / "alternative_statistics.csv")
slide.to_csv(OUT / "span_cut_sensitivity.csv", index=False)
print("wrote:", sorted(p.name for p in OUT.iterdir()))